# 10v5_03 — TL regime audit (no training)

**Questions this notebook measures (none of them is assumed):**

1. How often does the TRUE candidate of a TL_EVAL query have at least one eligible reference, under
   `standard` and under `mirror_aware`?
2. Does TL_EVAL mix Mode-A queries (truth has reference evidence) with reference-absent ones, and
   how does that compare with HOST?
3. Do test-like timsTOF spectra from Enveda-owned libraries lose their truth references mainly
   because `mirror_aware` excludes same-library evidence? *This is a question. It is answered
   by the source-library breakdown below.*

**Protocols vs the hidden test**

| protocol | allows | reading |
|---|---|---|
| `standard` | same-source / same-library references (only the query itself is excluded) | optimistic for a benchmark query whose library mates were measured in the same session |
| `mirror_aware` | no same-library references and no T1/T2 mirrors | pessimistic when the hidden test shares an instrument or platform, but not the session, with training libraries |
| hidden test | unpublished query spectra; training references may come from the same instrument or platform, never from the hidden query's own session | lies somewhere between the two; neither protocol is claimed to be the exact hidden-test regime |

So this audit reports a **diagnostic range**, not a test MRR forecast. It consumes existing QCR
evidence only (no similarity is recomputed, and nothing is trained). A dataset whose QCR has not
been built yet is reported as `NOT_BUILT`.

Regimes: `MODE_A_MIRROR`, `STANDARD_ONLY`, `REF_ABSENT`, `POOL_ABSENT`, and `INCONSISTENT`, which
should be empty by construction and is asserted. **Routing (v5.2)**: the largest of MODE_A_MIRROR / STANDARD_ONLY / REF_ABSENT decides (MODE_A / RESOLVE_TEST_PROTOCOL /
REFERENCE_ABSENT). The v5.1 rule below is still recorded as `legacy_v5_1_route`. Legacy v5.1 rule, fixed before execution:
if the TL_EVAL `MODE_A_MIRROR` share is below **0.50**, the route is
`PRIORITIZE_REFERENCE_ABSENT`, otherwise `CONTINUE_MODE_A_SEQUENCE`. It is **not** a gate:
every scale build still finishes.

In [ ]:
import sys, json
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd

from casmi.paths import get_project_paths
from casmi.qcr.context import HOST_INGEST_LIB, v4b_paths, v5_paths
from casmi.qcr.settlement import load_settlement_or_refuse
from casmi.ranking.manifests import load_manifest, save_manifest
from casmi.ranking.rank_eval import summarize
from casmi.ranking.scaling import pool_size_stratification
from casmi.ranking.training_sets import MODE_A_SUFFIX, derive_mode_a_manifests
from casmi.validation.regime_audit import (
    AUDIT_PROTOCOLS, INCONSISTENT, MODE_A_MIRROR, NOT_BUILT, REF_ABSENT, REGIMES, ROUTE_REF_ABSENT, ROUTE_THRESHOLD, ArtifactNotBuilt,
    build_query_regimes, load_host_truth_evidence, load_v5_truth_evidence, metrics_by_regime, molecule_regime_mix, project_route,
    save_regimes, source_breakdown, summary_row, project_route_v2, ROUTE_V2_REF_ABSENT, ROUTE_V2_RESOLVE,
)

pd.set_option('display.max_columns', 60); pd.set_option('display.width', 220)
paths = get_project_paths()
vp, D = v4b_paths(paths), v5_paths(paths)
REG_DIR = D['root'] / 'regimes'
REG_DIR.mkdir(parents=True, exist_ok=True)
settlement = load_settlement_or_refuse(vp.settlement_json)
ARTS = settlement['evidence_artifacts']
UNITS = {'TL_EVAL': ['TL_EVAL'], 'TL_1K': ['TL_1K'], 'TL_3K': ['TL_1K', 'TL_3K'], 'TL_10K': ['TL_1K', 'TL_3K', 'TL_10K'],
         'RND_1K': ['RND_1K'], 'MOL_DEV': ['MOL_DEV']}
DATASETS = ['TL_EVAL', 'HOST', 'TL_1K', 'TL_3K', 'TL_10K', 'RND_1K', 'MOL_DEV']
print('protocols audited:', AUDIT_PROTOCOLS, '| routing threshold (pre-registered):', ROUTE_THRESHOLD)

## 1. Query-level regimes per dataset (existing QCR only; `NOT_BUILT` when absent)

In [ ]:
REG, STATUS = {}, {}
host_q = pd.read_parquet(vp.host_processed / 'host_holdout_spectra.parquet')
host_manifest = host_q.rename(columns={'true_connectivity_key': 'connectivity_key', 'ingest_lib': 'source', 'instrument_type': 'instrument'})
for name in DATASETS:
    try:
        if name == 'HOST':
            manifest = host_manifest
            pool, qcr_t, pairs_t = load_host_truth_evidence(ARTS)
        else:
            if not (D['manifests'] / f'{name}.parquet').exists():
                raise ArtifactNotBuilt(f'manifest {name} missing')
            manifest, _ = load_manifest(name, D['manifests'])
            pool, qcr_t, pairs_t = load_v5_truth_evidence(D, UNITS[name])
            pool = pool[pool['query_id'].isin(set(manifest['query_id']))]
        r = build_query_regimes(manifest, pool, qcr_t, pairs_t)
        assert len(r) == len(manifest) and not r['query_id'].duplicated().any()
        assert set(r['regime']) <= set(REGIMES)
        assert (r['regime'] == INCONSISTENT).sum() == 0, 'INCONSISTENT regime rows: standard must include every mirror_aware reference'
        meta = save_regimes(r, name, REG_DIR)
        REG[name], STATUS[name] = r, 'BUILT'
        print(f'{name}: {len(r):,} queries -> {meta["regime_counts"]}')
    except (ArtifactNotBuilt, FileNotFoundError) as e:
        STATUS[name] = NOT_BUILT
        print(f'{name}: {NOT_BUILT} ({e})')
print(STATUS)

## 2. Primary summary table (percent of each dataset's queries)

In [ ]:
rows = []
for name in DATASETS:
    if STATUS[name] != 'BUILT':
        rows.append({'dataset': name, 'status': NOT_BUILT})
        continue
    s = summary_row(REG[name])
    rows.append({'dataset': name, 'status': 'BUILT', 'n_queries': s['n_queries'], 'truth_in_pool_pct': 100 * s['truth_in_pool_share'],
                 **{f'{p}_ge{k}_pct': 100 * s[f'{p}_ge{k}_share'] for p in ('standard', 'mirror') for k in (1, 3, 5)},
                 'MODE_A_MIRROR_pct': 100 * s['mode_a_share'], 'STANDARD_ONLY_pct': 100 * s['standard_only_share'],
                 'REF_ABSENT_pct': 100 * s['ref_absent_share'], 'POOL_ABSENT_pct': 100 * s['pool_absent_share'],
                 'median_mirror_refs_censored': s['median_mirror_refs_censored']})
summary_table = pd.DataFrame(rows).set_index('dataset')
display(summary_table.round(2))
summary_table.to_csv(REG_DIR / 'regime_summary_table.csv')

## 3. Source-library breakdown (the same-library-exclusion question)

In [ ]:
SRC = {}
for name in DATASETS:
    if STATUS[name] == 'BUILT':
        SRC[name] = source_breakdown(REG[name])
        SRC[name].to_csv(REG_DIR / f'{name}_source_breakdown.csv', index=False)
        print(f'\n== {name}')
        cols = ['source', 'n_queries', 'truth_in_pool_share', 'standard_ge1_share', 'standard_ge3_share', 'standard_ge5_share', 'mirror_ge1_share',
                'mirror_ge3_share', 'mirror_ge5_share', 'mode_a_share', 'standard_only_share', 'ref_absent_share', 'pool_absent_share']
        display(SRC[name][cols].sort_values('n_queries', ascending=False).round(3))

## 4. Standard vs mirror_aware diagnostic (TL_EVAL, HOST). Diagnostic only; no protocol switch, no selection.

This uses the V1-compatible spectral-feature coverage: whether a candidate has at least one
reference selected by the protocol, i.e. whether its 8 spectral features are finite. It is
measured for truth and decoy candidates, from the existing QCR `accepted_<protocol>` flags.

In [ ]:
def coverage_rows(name, qcr_paths, pool):
    parts = []
    for p in qcr_paths:
        q = pd.read_parquet(p, columns=['query_id', 'candidate_key', 'accepted_standard', 'accepted_mirror_aware'])
        parts.append(q.groupby(['query_id', 'candidate_key'])[['accepted_standard', 'accepted_mirror_aware']].any().reset_index())
    cov = pd.concat(parts, ignore_index=True).groupby(['query_id', 'candidate_key']).any().reset_index()
    d = pool.merge(cov.rename(columns={'candidate_key': 'candidate_connectivity_key'}), on=['query_id', 'candidate_connectivity_key'], how='left')
    d[['accepted_standard', 'accepted_mirror_aware']] = d[['accepted_standard', 'accepted_mirror_aware']].fillna(False).astype(bool)
    out = []
    for role, g in (('truth', d[d['is_true_candidate']]), ('decoy', d[~d['is_true_candidate']])):
        out.append({'dataset': name, 'candidate_role': role, 'n_candidates': int(len(g)),
                    'coverage_standard': float(g['accepted_standard'].mean()), 'coverage_mirror_aware': float(g['accepted_mirror_aware'].mean()),
                    'available_to_absent_share': float((g['accepted_standard'] & ~g['accepted_mirror_aware']).mean())})
    return out


cov_rows = []
if STATUS['TL_EVAL'] == 'BUILT':
    cov_rows += coverage_rows('TL_EVAL', sorted((D['qcr'] / 'TL_EVAL' / 'qcr' / 'shards').glob('part-*.parquet')),
                              pd.read_parquet(D['features'] / 'units' / 'TL_EVAL_pool.parquet'))
cov_rows += coverage_rows('HOST', [ARTS['host_qcr']], pd.read_parquet(ARTS['host_features_mirror_aware'], columns=['query_id', 'candidate_connectivity_key', 'is_true_candidate']))
coverage = pd.DataFrame(cov_rows)
coverage.to_csv(REG_DIR / 'standard_vs_mirror_coverage.csv', index=False)
display(coverage.round(4))

## 5. TL_EVAL performance by regime (existing provisional 1k predictions; nothing trained or tuned)

Rows show all queries (`ALL`, with pool-absent queries at RR = 0) and then each regime. V1 is
compared with B1 (mass only), which separates missing reference evidence from generally dense
pools. B5 is the frozen v4b configuration, with nothing tuned on TL_EVAL.

In [ ]:
PERF = {}
if STATUS['TL_EVAL'] == 'BUILT':
    for label, fname in (('V1_TL_1K', 'tl_eval_per_query_V1_TL_1K'), ('V1_RND_1K', 'tl_eval_per_query_V1_RND_1K'),
                         ('B1_mass', 'tl_eval_per_query_B1_mass'), ('B5_frozen_v4b', 'tl_eval_per_query_B5_frozen_v4b')):
        p = D['metrics'] / f'{fname}.parquet'
        if p.exists():
            PERF[label] = metrics_by_regime(pd.read_parquet(p), REG['TL_EVAL']).assign(model=label)
        else:
            print(f'{label}: prediction table not found ({p.name}) -- run 10v5_02 at 1k first')
    perf = pd.concat(PERF.values(), ignore_index=True) if PERF else pd.DataFrame()
    if len(perf):
        display(perf.pivot_table(index=['population'], columns='model', values='mrr_at_25').round(4))
        display(perf[['model', 'population', 'n_queries', 'mrr_at_25', 'hit_at_1', 'hit_at_5', 'hit_at_25']].round(4))
        perf.to_csv(REG_DIR / 'tl_eval_metrics_by_regime.csv', index=False)

## 6. Candidate density by regime (TL_EVAL; n shown)

In [ ]:
if STATUS['TL_EVAL'] == 'BUILT' and (D['metrics'] / 'tl_eval_per_query_V1_TL_1K.parquet').exists():
    pq = pd.read_parquet(D['metrics'] / 'tl_eval_per_query_V1_TL_1K.parquet').merge(REG['TL_EVAL'][['query_id', 'regime']], on='query_id')
    dens = pd.concat([pool_size_stratification(pq[pq['regime'] == g]).assign(regime=g) for g in (MODE_A_MIRROR, REF_ABSENT)]
                     + [pool_size_stratification(pq).assign(regime='ALL')], ignore_index=True)
    display(dens)
    dens.to_csv(REG_DIR / 'tl_eval_density_by_regime_V1_TL_1K.csv', index=False)

## 7. Mode-A training manifests (derived; the originals are never modified)

A training query is kept iff its truth is in the pool and has at least one eligible
`mirror_aware` reference. The filter is a per-query decision, so
`TL_1K_MODE_A ⊂ TL_3K_MODE_A ⊂ TL_10K_MODE_A` is asserted and never repaired by resampling.
No QCR is rebuilt. No regime, reference-count or availability value is ever a model feature.

In [ ]:
train_names = [n for n in ('TL_1K', 'TL_3K', 'TL_10K', 'RND_1K') if STATUS.get(n) == 'BUILT']
originals = {n: load_manifest(n, D['manifests'])[0] for n in train_names}
before_sha = {n: load_manifest(n, D['manifests'])[1]['manifest_sha256'] for n in train_names}
mode_a, audits = derive_mode_a_manifests(originals, {n: REG[n] for n in train_names})
tl_keys, host_keys = set(load_manifest('TL_EVAL', D['manifests'])[0]['connectivity_key']), set(host_q['true_connectivity_key'])
for name, m in mode_a.items():
    assert (m['query_id'].map(REG[name.replace(MODE_A_SUFFIX, '')].set_index('query_id')['regime']) == MODE_A_MIRROR).all()
    assert not (set(m['connectivity_key']) & tl_keys) and not (set(m['connectivity_key']) & host_keys)
    assert HOST_INGEST_LIB not in set(m['source'])
    save_manifest(m, name, D['manifests'], extra={'derived_from': name.replace(MODE_A_SUFFIX, ''), **audits[name]})
after_sha = {n: load_manifest(n, D['manifests'])[1]['manifest_sha256'] for n in train_names}
assert before_sha == after_sha, 'an original manifest changed'
retention = pd.DataFrame(audits).T[['n_original', 'n_mode_a_retained', 'retention_rate']]
display(retention)

# missingness-shortcut diagnostic (audit only): every retained training query's truth has a mirror reference;
# decoy no-reference rates are reported, never used as a feature
diag = []
for name in mode_a:
    base = name.replace(MODE_A_SUFFIX, '')
    fp = D['features'] / f'{base}_mirror_aware.parquet'
    if not fp.exists():
        continue
    f = pd.read_parquet(fp, columns=['query_id', 'is_true_candidate', 'n_reference_spectra'])
    f = f[f['query_id'].isin(set(mode_a[name]['query_id']))]
    truth_ok = f[f['is_true_candidate']].groupby('query_id')['n_reference_spectra'].min().ge(1)
    assert truth_ok.all() and truth_ok.size == mode_a[name]['query_id'].nunique(), f'{name}: a retained truth lacks a mirror_aware reference'
    diag.append({'manifest': name, 'n_queries': int(truth_ok.size), 'truth_has_mirror_reference_share': float(truth_ok.mean()),
                 'decoy_no_reference_rate': float((f.loc[~f['is_true_candidate'], 'n_reference_spectra'] == 0).mean())})
if diag:
    display(pd.DataFrame(diag))

## 8. MOL_DEV molecule regime mix (preparation for aggregation; not used for any selection)

In [ ]:
if STATUS.get('MOL_DEV') == 'BUILT':
    mix = molecule_regime_mix(REG['MOL_DEV'])
    mix.to_parquet(REG_DIR / 'MOL_DEV_molecule_regime_mix.parquet', index=False)
    display(mix['molecule_regime_mix'].value_counts().rename('n_molecules').to_frame())
else:
    print('MOL_DEV: NOT_BUILT -- rerun this notebook after scripts/v5_build_scale_features.py --manifest MOL_DEV')

## 9. Summary JSON, project route and decision text (all computed at runtime)

In [ ]:
def summary_fields(name):
    if STATUS.get(name) != 'BUILT':
        return {'status': NOT_BUILT}
    s = summary_row(REG[name])
    keys = ['n_queries', 'truth_in_pool_share', 'standard_ge1_share', 'mirror_ge1_share', 'mirror_ge3_share', 'mirror_ge5_share', 'mode_a_share',
            'standard_only_share', 'ref_absent_share', 'pool_absent_share', 'median_mirror_refs_censored']
    return {'status': 'BUILT', **{k: s[k] for k in keys}}


if STATUS['TL_EVAL'] == 'BUILT':
    _s = summary_row(REG['TL_EVAL'])
    route = {**project_route_v2(_s['mode_a_share'], _s['standard_only_share'], _s['ref_absent_share']),
             'legacy_v5_1_route': project_route(_s['mode_a_share'])['project_route']}   # recorded, superseded (cannot separate protocol-induced absence)
else:
    route = {'project_route': NOT_BUILT}
model_metrics = {}
if 'V1_TL_1K' in PERF:
    p = PERF['V1_TL_1K'].set_index('population')
    model_metrics = {f'TL_EVAL_{k}': p.loc[v].drop('model').to_dict() for k, v in
                     (('ALL', 'ALL'), ('MODE_A', MODE_A_MIRROR), ('STANDARD_ONLY', 'STANDARD_ONLY'), ('REF_ABSENT', REF_ABSENT)) if v in p.index}
audit_summary = {'status': STATUS, **{n: summary_fields(n) for n in DATASETS},
                 'training': {n: audits[n] for n in audits}, 'model_metrics_V1_TL_1K_provisional': model_metrics,
                 'coverage_standard_vs_mirror': cov_rows, 'project_route': route['project_route']}
(D['root'] / 'regime_audit_summary.json').write_text(json.dumps(audit_summary, indent=2, default=str), encoding='utf-8')
(D['root'] / 'project_route.json').write_text(json.dumps(route, indent=2, default=str), encoding='utf-8')

if STATUS['TL_EVAL'] == 'BUILT':
    s = summary_row(REG['TL_EVAL'])
    print(f"TL-EVAL mirror-aware Mode-A coverage: {100 * s['mode_a_share']:.1f}%")
    print(f"TL-EVAL standard-only coverage:       {100 * s['standard_only_share']:.1f}%")
    print(f"TL-EVAL fully reference-absent:       {100 * s['ref_absent_share']:.1f}%")
    print(f"TL-EVAL truth absent from pool:       {100 * s['pool_absent_share']:.1f}%")
    print(f"\nPROJECT ROUTE (v5.2 rule): {route['project_route']} (dominant regime {route['dominant_regime']}; legacy v5.1 route {route['legacy_v5_1_route']})")
    if route['project_route'] == ROUTE_V2_RESOLVE:
        print('Next: 10v5_04_standard_only_provenance.ipynb, then 10v5_05_test_simulated_1k_pilot.ipynb (3k/10k gated on its SCALE_GO)')
    if route['project_route'] == ROUTE_V2_REF_ABSENT:
        print('Next research notebook after the freeze: reference-absent route -- see docs/ROADMAP_REFERENCE_ABSENT.md')